# Frontend Pipeline For Llama 405B

实现 Llama 405B decoder layer 的最小前端全流程，并在 `H100_SXM` 上完成 macro op 仿真。

## 流程
- 加载 `llama-405B` model card 并初始化 `LlamaModelConfig`
- 初始化 `NandConfig`、`InferenceConfig`
- 初始化 `LlamaDecoderLayer`
- 使用 `NxTracer` trace 得到计算图
- 运行 `NormalizePass` 获得简化图
- 用 `build_kv_cache_state` 构造 `KVCacheState`
- 构造 `NxGraphMeta` 并写入 `graph.meta`
- 运行 `CodeGenPass` 生成 `macro_op_list`
- 在 `H100_SXM` 上运行完整硬件仿真并校验最终 cycle

In [1]:
import json
import logging
from collections import Counter
from pathlib import Path

import torch
from torch.fx import GraphModule

from nandmachine.commands.macro import FlashAttnOp, MatMulOp, VectorOp
from nandmachine.config.config import NandConfig
from nandmachine.config.inference_config import DenseParallelConfig, InferenceConfig
from nandmachine.config.model_config import LlamaModelConfig
from nandmachine.frontend.core.graph.base import NxGraphMeta, NxTracer
from nandmachine.frontend.core.passes.cod_gen import CodeGenPass
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.llama import LlamaDecoderLayer
from nandmachine.frontend.utlis import build_kv_cache_state
from nandmachine.simulator.entry_point import run_macro_ops

MODEL_CARD_PATH = Path("model_cards/llama-405B.json")
DEVICE_NAME = "H200_SXM"
COMPILE_MODE = "heuristic-GPU"
HBM_BANDWIDTH_GBPS = 3350

assert MODEL_CARD_PATH.exists(), MODEL_CARD_PATH


def node_summary(graph_module: GraphModule) -> list[tuple[str, str]]:
    return [(node.op, str(node.target)) for node in graph_module.graph.nodes]


def macro_summary(macro_op_list: list[object]) -> Counter:
    return Counter(type(op).__name__ for op in macro_op_list)


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

In [3]:
model_config = LlamaModelConfig.from_dict(json.loads(MODEL_CARD_PATH.read_text()))
nand_config = NandConfig(
    num_channels=6,
    num_plane=96,
    num_block=16,
    num_pages=16,
    tRead=4000, # ns
    tWrite=4000 * 10,
    tErase=4000 * 100,
    page_size=4, # KB
    sram_threshold=1024 * 80, # KB
)
inference_config = InferenceConfig(
    batch_size=128,
    input_sequence_length=8096,
    output_sequence_length=1024,
    weight_bits=16,
    activation_bits=16,
    kv_cache_bits=16,
    kv_block_size_bytes= 1024 * 256,
    memory_backend="nand",
    parallel_config=DenseParallelConfig(num_ranks=8, tp_size=8, dp_size=1),
)
kv_cache_state = build_kv_cache_state(nand_config, model_config, inference_config)
graph_meta = NxGraphMeta(
    nand_config=nand_config,
    model_config=model_config,
    inference_config=inference_config,
    kv_cache_state=kv_cache_state,
)
simulator_config = {
    "device_name": DEVICE_NAME,
    "compile_mode": COMPILE_MODE,
    "hbm_bandwidth_GBps": HBM_BANDWIDTH_GBPS,
}
runtime_simulator_config = {
    "device_name": simulator_config["device_name"],
    "compile_mode": simulator_config["compile_mode"],
    "hbm_bandwidth_bytes_per_sec": simulator_config["hbm_bandwidth_GBps"] * 10**9,
}


LlamaModelConfig(attention_type='gqa', hidden_size=16384, num_attention_heads=128, num_key_value_heads=8, max_position_embeddings=131072, intermediate_size=53248, hidden_act='silu', head_dim=128, rms_norm_eps=1e-05, attention_bias=False, mlp_bias=False, rope_theta=500000.0, rope_scaling={'factor': 8.0, 'low_freq_factor': 1.0, 'high_freq_factor': 4.0, 'original_max_position_embeddings': 8192, 'rope_type': 'llama3'}, num_hidden_layers=126)
InferenceConfig(batch_size=128, input_sequence_length=8096, output_sequence_length=1024, weight_bits=16, activation_bits=16, kv_cache_bits=16, kv_block_size_bytes=262144, memory_backend='nand', parallel_config=DenseParallelConfig(num_ranks=8, tp_size=8, dp_size=1))
KVCacheState(total_kv_cache_size_per_layer=597688320, num_nand_pages_per_layer=145920, num_hyper_pages_per_layer=1520, kv_block_size_tokens=512, num_kv_blocks=2280, kv_cache_num_pages_per_layer=145920)
{'device_name': 'H100_SXM', 'compile_mode': 'heuristic-GPU'}


In [6]:
with torch.device("meta"):
    model = LlamaDecoderLayer(model_config,inference_config.parallel_config.tp_size)
    graph = NxTracer().trace(model)
    graph_module = GraphModule(model, graph)

NormalizePass().transform(graph_module)
graph_module.graph.meta = {CodeGenPass.GRAPH_META_KEY: graph_meta}
CodeGenPass().transform(graph_module)

macro_op_list = graph_module.graph.meta[CodeGenPass.MACRO_OP_LIST_META_KEY]
vector_ops = [op for op in macro_op_list if isinstance(op, VectorOp)]

print("normalized node count:", len(node_summary(graph_module)))
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("first 10 vector ops:", [(op.vector_op_type, op.vector_shape) for op in vector_ops[:10]])


2026-04-01 21:13:44,396 - INFO - Processed node=input_layernorm generated_macro_ops=1
2026-04-01 21:13:44,397 - INFO - Processed node=self_attn_qkv_proj generated_macro_ops=3
2026-04-01 21:13:44,397 - INFO - Processed node=self_attn_rotary_emb generated_macro_ops=0
2026-04-01 21:13:44,401 - INFO - Processed node=self_attn_attn generated_macro_ops=570
2026-04-01 21:13:44,403 - INFO - Processed node=self_attn_o_proj generated_macro_ops=4
2026-04-01 21:13:44,403 - INFO - Processed node=post_attention_layernorm generated_macro_ops=1
2026-04-01 21:13:44,404 - INFO - Processed node=mlp_gate_up_proj generated_macro_ops=18


2026-04-01 21:13:44,405 - INFO - Processed node=mlp_act_fn generated_macro_ops=1
2026-04-01 21:13:44,407 - INFO - Processed node=mlp_down_proj generated_macro_ops=10


normalized node count: 13
macro op count: 608
macro op type summary: {'VectorOp': 3, 'SramPrefetch': 201, 'MatMulOp': 11, 'SramPrefetchRelease': 201, 'FlashAttnOp': 190, 'AllReduceOp': 2}
first 10 vector ops: [('rms_norm', [128, 16384]), ('rms_norm', [128, 16384]), ('silu_mul', [128, 6656])]


In [7]:
assert macro_op_list
assert any(isinstance(op, MatMulOp) for op in macro_op_list)
assert any(isinstance(op, FlashAttnOp) for op in macro_op_list)
assert any(op.vector_op_type == "rms_norm" for op in vector_ops)
assert any(op.vector_op_type == "silu_mul" for op in vector_ops)
assert all(all(dim > 0 for dim in op.vector_shape) for op in vector_ops)

print("llama frontend pipeline completed successfully")

llama frontend pipeline completed successfully


In [ ]:
macro_op_list[:50]

[VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 16384], weight_bits=16),
 SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432),
 MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432)], dim=(128, 16384, 2304), weight_bits=16),
 SramPrefetchRelease(id=4, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432)], dim=(128, 16384, 2304), weight_bits=16)]),
 SramPrefetch(id=5, input_ops=[], num_prefetch_pages=96),
 FlashAttnOp(id=6, input_ops=[SramPrefetch(id=5, input_ops=[], num_prefetch_pages=96)], qk_bmm_shape=(1, 16, 128, 512), sv_bmm_shape=(1, 16, 512, 128), softmax_shape=(16, 512), weight_bits=16),
 SramPrefetchRelease(id=7, input_ops=[FlashAttnOp(id=6, input_ops=[SramPrefetch(id=5, input_ops=[], num_prefetch_pages=96)], qk_bmm_shape=(1, 16, 128, 512), sv_bmm_shape=(1, 16, 512, 128), softmax_shape=(16, 512), weight_bits=16)]),
 SramPrefetch(id=8, input_ops=[], num_prefetch_pages=96)

In [9]:
len(macro_op_list)

608

## 完整硬件仿真

直接复用前面生成好的 `nand_config`、`inference_config`、`kv_cache_state` 和 `macro_op_list`，在 `H100_SXM` 上运行完整 macro op 仿真。

In [ ]:
sim_result = run_macro_ops(
    nand_config,
    macro_op_list,
    **runtime_simulator_config,
)

assert sim_result.cycle > 0
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("nand config:", nand_config)
print("inference config:", inference_config)
print("kv cache state:", kv_cache_state)
print("HBM bandwidth (GBps):", HBM_BANDWIDTH_GBPS)
print("simulator config:", simulator_config)
print("final sim cycle:", sim_result.cycle)
print("final sim time ns:", sim_result.time_ns)
